# Where Pipelines Disagree: Does Analytical Uncertainty Concentrate in Clinically Important Brain Regions?

## A Novel MI-CDM-Enabled Analysis of Tau PET Processing Pipelines in the A4/LEARN Trial

**Motivation**: In Alzheimer's clinical trials, tau PET images are processed by different computational pipelines (e.g., PetSurfer, Stanford) that produce different regional SUVR values. Some brain regions are more prognostic of cognitive decline than others. A critical methodological question has not been addressed:

> **Does pipeline disagreement concentrate in the brain regions that matter most for predicting clinical outcomes?**

If high-prognostic regions have *low* pipeline agreement, then pipeline choice directly affects clinical inferences. If pipeline agreement is uniform or higher in prognostic regions, pipeline choice is less consequential for clinical endpoints.

**Why MI-CDM is required**: This analysis depends on two capabilities provided by the MI-CDM extension (Park et al. 2025, J Imaging Inform Med):
1. **`image_feature.alg_system`** identifies which processing pipeline produced each measurement
2. **`image_occurrence.image_study_UID`** ensures cross-pipeline comparison uses measurements from the *same physical scan*, not date-matching heuristics

Without MI-CDM, this would require fragile string-parsing of `measurement_source_value` AND approximate date matching — error-prone and not reproducible.

*Based on the MI-CDM extension (Park et al. 2025, J Imaging Inform Med, 37:899-908).*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
sns.set_theme(style='whitegrid')

BASE_DIR = Path('..')
OMOP_DIR = BASE_DIR / 'OMOP_Output'
MI_CDM_DIR = OMOP_DIR / 'mi_cdm'
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

# ── Standard OMOP tables ──
person = pd.read_csv(OMOP_DIR / 'person.csv')
measurement = pd.read_csv(OMOP_DIR / 'measurement.csv', low_memory=False)

# ── MI-CDM extension tables (Park et al. 2025) ──
image_occurrence = pd.read_csv(MI_CDM_DIR / 'image_occurrence.csv')
image_feature = pd.read_csv(MI_CDM_DIR / 'image_feature.csv')
procedure_occurrence = pd.read_csv(MI_CDM_DIR / 'procedure_occurrence.csv')

print('Data loaded.')
print('{:>30s}: {:>10,} rows'.format('measurement', len(measurement)))
print('{:>30s}: {:>10,} rows'.format('procedure_occurrence', len(procedure_occurrence)))
print('{:>30s}: {:>10,} rows'.format('image_occurrence', len(image_occurrence)))
print('{:>30s}: {:>10,} rows'.format('image_feature', len(image_feature)))

In [ ]:
# ── Helper Functions ──

def get_imaging_measurements_by_pipeline(alg_system_pattern):
    """Get measurements linked to a specific processing pipeline via MI-CDM.

    Join path: image_feature (filter by alg_system) -> measurement
    Also attaches image_occurrence_id and image_study_UID for cross-pipeline scan matching.
    """
    feats = image_feature[
        image_feature['alg_system'].str.contains(alg_system_pattern, case=False, na=False)
    ][['image_feature_event_id', 'image_occurrence_id', 'alg_system']]
    meas = measurement[measurement['measurement_id'].isin(feats['image_feature_event_id'])].copy()
    meas = meas.merge(
        feats, left_on='measurement_id', right_on='image_feature_event_id', how='left'
    )
    # Attach image_study_UID for cross-pipeline matching
    meas = meas.merge(
        image_occurrence[['image_occurrence_id', 'image_study_UID']],
        on='image_occurrence_id', how='left'
    )
    return meas


def get_baseline(df, person_col='person_id', date_col='measurement_date'):
    """Get earliest record per person."""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    idx = df.groupby(person_col)[date_col].idxmin()
    return df.loc[idx]


def icc_2_1(y1, y2):
    """ICC(2,1): two-way random, single measures, absolute agreement."""
    n = len(y1)
    if n < 3:
        return np.nan
    data = np.column_stack([y1, y2])
    k = 2
    row_means = data.mean(axis=1)
    col_means = data.mean(axis=0)
    grand_mean = data.mean()
    SSR = k * np.sum((row_means - grand_mean) ** 2)
    SSC = n * np.sum((col_means - grand_mean) ** 2)
    SST = np.sum((data - grand_mean) ** 2)
    SSE = SST - SSR - SSC
    MSR = SSR / (n - 1) if n > 1 else 0
    MSC = SSC / (k - 1) if k > 1 else 0
    MSE = SSE / ((n - 1) * (k - 1)) if (n - 1) * (k - 1) > 0 else 0
    denom = MSR + (k - 1) * MSE + k * (MSC - MSE) / n
    if denom <= 0:
        return 0.0
    return (MSR - MSE) / denom

print('Helpers loaded.')

---
## MI-CDM Data Architecture (Park et al. 2025)

The MI-CDM extends OMOP CDM v5.4 with **two new tables**:

```
Procedure_Occurrence (standard OMOP — one per imaging session)
    -> Image_Occurrence (one per DICOM series)
        -> Image_Feature (polymorphic bridge → clinical domain tables)
            -> Measurement (standard OMOP measurement table)
```

Key capabilities for this analysis:
- **`image_feature.alg_system`**: URN identifying the processing pipeline (e.g., `urn:a4:pipeline:petsurfer`)
- **`image_occurrence.image_study_UID`**: Synthetic DICOM Study UID linking all series from the same scan session
- **Polymorphic event pattern**: `image_feature_event_field_concept_id` (1147330 = measurement.measurement_id) + `image_feature_event_id` (the actual measurement_id)

In [ ]:
# ── Processing Pipeline Landscape ──
# Use image_feature.alg_system to show which pipelines are active
feat_with_io = image_feature.merge(
    image_occurrence[['image_occurrence_id', 'modality_concept_id']],
    on='image_occurrence_id'
)

# Map modality concepts to names
modality_names = {2100000090: 'MR', 2100000091: 'PT', 2100000092: 'OP'}
feat_with_io['modality'] = feat_with_io['modality_concept_id'].map(modality_names)

# Short pipeline names
feat_with_io['pipeline'] = feat_with_io['alg_system'].str.replace(
    'urn:a4:pipeline:', '', regex=False)

pipeline_matrix = pd.crosstab(feat_with_io['modality'], feat_with_io['pipeline'])

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pipeline_matrix, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Image Feature Records'})
ax.set_title('MI-CDM Processing Pipeline Landscape\n'
             'Each tau PET scan is processed by 3 independent pipelines',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Modality')
ax.set_xlabel('Processing Pipeline (alg_system)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('results/micdm_pipeline_landscape.png', bbox_inches='tight', dpi=150)
plt.show()

# Show tau-specific pipeline distribution
tau_feat = feat_with_io[feat_with_io['pipeline'].isin(['suvr_tau', 'petsurfer', 'stanford'])]
print('\nTau PET pipeline feature counts:')
print(tau_feat['pipeline'].value_counts().to_string())
tau_io_ids = tau_feat['image_occurrence_id'].unique()
print(f'\nUnique tau PET image_occurrences: {len(tau_io_ids)}')

---
## The Novel Question

PetSurfer and Stanford share **42 bilateral FreeSurfer brain regions** with exact name matching. For each region, we can compute two independent quantities:

1. **Pipeline Agreement (ICC)**: How well do PetSurfer and Stanford agree on the SUVR value for that region? Computed from same-scan matched pairs via `image_study_UID`.

2. **Prognostic Value (|*r*|)**: How strongly does the baseline SUVR in that region correlate with the rate of longitudinal cognitive decline (PACC slope)?

By plotting these two quantities against each other — one point per brain region — we can answer:

> **Do the regions where pipelines disagree most also happen to be the regions that matter most for predicting cognitive decline?**

This analysis is only possible with MI-CDM because:
- ICC computation requires matching PetSurfer and Stanford measurements from the **same physical scan** (via `image_study_UID`)
- Pipeline-specific extraction requires `image_feature.alg_system` to identify provenance

In [ ]:
# ── Step 1: Cross-Pipeline ICC per Region ──
# Extract pipeline-specific measurements via MI-CDM

ps_meas = get_imaging_measurements_by_pipeline('petsurfer')
sf_meas = get_imaging_measurements_by_pipeline('stanford')

# Parse region names
ps_meas['region'] = ps_meas['measurement_source_value'].str.replace(
    'TAU_PETSURFER:', '', regex=False)
sf_meas['region'] = sf_meas['measurement_source_value'].str.replace(
    'TAU_STANFORD:', '', regex=False)

# Find shared regions
shared_regions = sorted(set(ps_meas['region'].unique()) & set(sf_meas['region'].unique()))
print('Shared regions between PetSurfer and Stanford: {}'.format(len(shared_regions)))

# Match on image_study_UID + region (same physical scan, same brain region)
ps_shared = ps_meas[ps_meas['region'].isin(shared_regions)].copy()
sf_shared = sf_meas[sf_meas['region'].isin(shared_regions)].copy()
matched = ps_shared.merge(sf_shared, on=['image_study_UID', 'region'],
                          suffixes=('_ps', '_sf'))
matched = matched.dropna(subset=['value_as_number_ps', 'value_as_number_sf'])
print('Matched measurement pairs: {:,}'.format(len(matched)))
print('Unique subjects: {}'.format(matched['person_id_ps'].nunique()))

# AD-relevant regions
AD_REGIONS = [
    'bi_entorhinal', 'bi_Hippocampus', 'bi_Amygdala', 'bi_fusiform',
    'bi_inferiortemporal', 'bi_middletemporal', 'bi_parahippocampal',
    'bi_inferiorparietal', 'bi_isthmuscingulate', 'bi_posteriorcingulate',
    'bi_precuneus', 'bi_superiortemporal', 'bi_lateralorbitofrontal',
    'bi_insula',
]
ad_shared = [r for r in AD_REGIONS if r in shared_regions]

# Compute ICC per region
icc_results = {}
for region in shared_regions:
    rd = matched[matched['region'] == region]
    if len(rd) < 10:
        continue
    ps_vals = rd['value_as_number_ps'].values.astype(float)
    sf_vals = rd['value_as_number_sf'].values.astype(float)
    icc_results[region] = {
        'icc': icc_2_1(ps_vals, sf_vals),
        'n_pairs': len(rd),
        'ad_relevant': region in ad_shared,
    }

print('ICC computed for {} regions'.format(len(icc_results)))
print('Median ICC: {:.3f}'.format(
    np.median([v['icc'] for v in icc_results.values()])))

In [ ]:
# ── Step 2: Compute Longitudinal PACC Slope per Person ──
# PACC slope = rate of cognitive change (negative = decline)

pacc = measurement[measurement['measurement_source_value'] == 'PACC:PACC.raw'].copy()
pacc['value_as_number'] = pd.to_numeric(pacc['value_as_number'], errors='coerce')
pacc['measurement_date'] = pd.to_datetime(pacc['measurement_date'])
pacc = pacc.dropna(subset=['value_as_number', 'measurement_date'])

# Compute years from first visit
bl_dates = pacc.groupby('person_id')['measurement_date'].min().rename('bl_date')
pacc = pacc.merge(bl_dates, on='person_id')
pacc['years'] = (pacc['measurement_date'] - pacc['bl_date']).dt.days / 365.25

# Require 2+ visits for slope estimation
visit_counts = pacc.groupby('person_id').size()
valid_persons = visit_counts[visit_counts >= 2].index
pacc = pacc[pacc['person_id'].isin(valid_persons)]

# Linear regression per person: PACC ~ years
slopes = {}
for pid, grp in pacc.groupby('person_id'):
    if len(grp) >= 2 and grp['years'].std() > 0:
        slope, _, _, _, _ = stats.linregress(grp['years'].values,
                                              grp['value_as_number'].values)
        if np.isfinite(slope):
            slopes[pid] = slope

pacc_slopes = pd.Series(slopes, name='pacc_slope')
print('PACC slopes computed for {} persons'.format(len(pacc_slopes)))
print('Mean slope: {:.3f} (SD={:.3f})'.format(pacc_slopes.mean(), pacc_slopes.std()))
print('Negative slopes (declining): {} ({:.1f}%)'.format(
    (pacc_slopes < 0).sum(), 100 * (pacc_slopes < 0).mean()))

# How many tau PET patients have PACC slopes?
tau_persons = set(ps_meas['person_id'].unique())
overlap = tau_persons & set(pacc_slopes.index)
print('\nTau PET patients with PACC slopes: {}'.format(len(overlap)))

In [ ]:
# ── Step 3: Per-Region Prognostic Value ──
# For each region, correlate baseline tau SUVR with PACC slope.
# Higher |r| = more prognostic region.

def compute_regional_prognostic(meas_df, pacc_slopes, regions):
    """Compute per-region correlation between baseline tau SUVR and PACC slope."""
    results = {}
    for region in regions:
        region_meas = meas_df[meas_df['region'] == region].copy()
        region_meas['value_as_number'] = pd.to_numeric(
            region_meas['value_as_number'], errors='coerce')
        if len(region_meas) == 0:
            continue
        bl = get_baseline(region_meas)
        bl_vals = bl.set_index('person_id')['value_as_number']
        merged = pd.DataFrame({
            'tau': bl_vals,
            'pacc_slope': pacc_slopes
        }).dropna()
        merged = merged[np.isfinite(merged['tau']) & np.isfinite(merged['pacc_slope'])]
        if len(merged) < 20:
            continue
        r, p = pearsonr(merged['tau'].values, merged['pacc_slope'].values)
        if np.isfinite(r):
            results[region] = {'r': r, 'p': p, 'n': len(merged), 'abs_r': abs(r)}
    return results

ps_prognostic = compute_regional_prognostic(ps_meas, pacc_slopes, list(icc_results.keys()))
print('PetSurfer: prognostic value computed for {} regions'.format(len(ps_prognostic)))

sf_prognostic = compute_regional_prognostic(sf_meas, pacc_slopes, list(icc_results.keys()))
print('Stanford:  prognostic value computed for {} regions'.format(len(sf_prognostic)))

common_regions = sorted(set(icc_results.keys()) & set(ps_prognostic.keys())
                        & set(sf_prognostic.keys()))
print('\nRegions with ICC + prognostic values from both pipelines: {}'.format(
    len(common_regions)))

region_df = pd.DataFrame([{
    'region': r,
    'icc': icc_results[r]['icc'],
    'n_pairs': icc_results[r]['n_pairs'],
    'ad_relevant': icc_results[r]['ad_relevant'],
    'ps_r': ps_prognostic[r]['r'],
    'ps_abs_r': ps_prognostic[r]['abs_r'],
    'ps_p': ps_prognostic[r]['p'],
    'ps_n': ps_prognostic[r]['n'],
    'sf_r': sf_prognostic[r]['r'],
    'sf_abs_r': sf_prognostic[r]['abs_r'],
    'sf_p': sf_prognostic[r]['p'],
    'sf_n': sf_prognostic[r]['n'],
    'mean_abs_r': (ps_prognostic[r]['abs_r'] + sf_prognostic[r]['abs_r']) / 2,
    'label': r.replace('bi_', ''),
} for r in common_regions])

region_df = region_df.dropna(subset=['icc', 'mean_abs_r', 'ps_abs_r', 'sf_abs_r'])
region_df = region_df[np.isfinite(region_df['icc']) & np.isfinite(region_df['mean_abs_r'])]
print('Regions after quality filter: {}'.format(len(region_df)))

print('\nTop 10 most prognostic regions (by mean |r| across both pipelines):')
top10 = region_df.nlargest(10, 'mean_abs_r')
for _, row in top10.iterrows():
    ad = '*' if row['ad_relevant'] else ' '
    print('  {} {:<25s}  |r|={:.3f}  ICC={:.3f}'.format(
        ad, row['label'], row['mean_abs_r'], row['icc']))

In [ ]:
# ── KEY FIGURE: Pipeline Agreement vs Prognostic Value ──

fig, ax = plt.subplots(figsize=(10, 8))

non_ad = region_df[~region_df['ad_relevant']]
ad = region_df[region_df['ad_relevant']]

ax.scatter(non_ad['mean_abs_r'], non_ad['icc'],
           s=80, c='#90CAF9', edgecolors='#1565C0', linewidth=0.8,
           alpha=0.8, label='Other regions (n={})'.format(len(non_ad)), zorder=3)
ax.scatter(ad['mean_abs_r'], ad['icc'],
           s=120, c='#FF8A65', edgecolors='#BF360C', linewidth=1.2,
           alpha=0.9, label='AD-relevant regions (n={})'.format(len(ad)), zorder=4,
           marker='D')

for _, row in ad.iterrows():
    ax.annotate(row['label'], (row['mean_abs_r'], row['icc']),
                fontsize=7.5, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points',
                fontstyle='italic', color='#BF360C')

top_non_ad = non_ad.nlargest(3, 'mean_abs_r')
for _, row in top_non_ad.iterrows():
    ax.annotate(row['label'], (row['mean_abs_r'], row['icc']),
                fontsize=7, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points',
                color='#1565C0')

fit_mask = np.isfinite(region_df['mean_abs_r']) & np.isfinite(region_df['icc'])
fit_df = region_df[fit_mask]
rho, p = spearmanr(fit_df['mean_abs_r'], fit_df['icc'])

try:
    z = np.polyfit(fit_df['mean_abs_r'].values, fit_df['icc'].values, 1)
    x_line = np.linspace(fit_df['mean_abs_r'].min(), fit_df['mean_abs_r'].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'k--', alpha=0.4, linewidth=1.5, zorder=2)
except np.linalg.LinAlgError:
    pass

textstr = 'Spearman rho = {:.3f}\np = {:.3f}\nn = {} regions'.format(rho, p, len(fit_df))
props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9)
ax.text(0.02, 0.02, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='bottom', bbox=props)

ax.axhline(y=0.75, color='green', linestyle=':', alpha=0.4, label='ICC = 0.75 (good)')
ax.axhline(y=0.50, color='orange', linestyle=':', alpha=0.4, label='ICC = 0.50 (moderate)')

ax.set_xlabel('Prognostic Value: |r(tau SUVR, PACC slope)|\n'
              '(averaged across PetSurfer and Stanford)', fontsize=11)
ax.set_ylabel('Pipeline Agreement: ICC(2,1)\n'
              '(PetSurfer vs Stanford, same scan)', fontsize=11)
ax.set_title('Does Pipeline Disagreement Concentrate\n'
             'in Clinically Important Brain Regions?',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.set_xlim(left=0)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('results/micdm_novel_agreement_vs_prognostic.png',
            bbox_inches='tight', dpi=150)
plt.show()

if p < 0.05:
    if rho < 0:
        print('FINDING: Significant NEGATIVE correlation (rho={:.3f}, p={:.3f}).'.format(rho, p))
        print('Pipeline disagreement is GREATER in more prognostic regions.')
    else:
        print('FINDING: Significant POSITIVE correlation (rho={:.3f}, p={:.3f}).'.format(rho, p))
        print('Pipeline disagreement is LOWER in more prognostic regions.')
else:
    print('FINDING: No significant correlation (rho={:.3f}, p={:.3f}).'.format(rho, p))
    print('Pipeline disagreement is distributed uniformly across prognostic and')
    print('non-prognostic regions.')

In [ ]:
# ── Do Both Pipelines Identify the Same Regions as Prognostic? ──

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
non_ad = region_df[~region_df['ad_relevant']]
ad_df = region_df[region_df['ad_relevant']]

ax.scatter(non_ad['ps_abs_r'], non_ad['sf_abs_r'],
           s=80, c='#90CAF9', edgecolors='#1565C0', linewidth=0.8,
           alpha=0.8, label='Other regions', zorder=3)
ax.scatter(ad_df['ps_abs_r'], ad_df['sf_abs_r'],
           s=120, c='#FF8A65', edgecolors='#BF360C', linewidth=1.2,
           alpha=0.9, label='AD-relevant', zorder=4, marker='D')

lim = max(region_df['ps_abs_r'].max(), region_df['sf_abs_r'].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, linewidth=1)

for _, row in ad_df.iterrows():
    ax.annotate(row['label'], (row['ps_abs_r'], row['sf_abs_r']),
                fontsize=7, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points',
                fontstyle='italic', color='#BF360C')

rho_prog, p_prog = spearmanr(region_df['ps_abs_r'], region_df['sf_abs_r'])
ax.set_xlabel('PetSurfer |r(tau, PACC slope)|', fontsize=11)
ax.set_ylabel('Stanford |r(tau, PACC slope)|', fontsize=11)
ax.set_title('Prognostic Ranking Agreement\nDo pipelines agree on which regions matter?',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
textstr = 'Spearman rho = {:.3f}\np = {:.1e}'.format(rho_prog, p_prog)
props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9)
ax.text(0.02, 0.95, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

ax2 = axes[1]
region_df['ps_rank'] = region_df['ps_abs_r'].rank(ascending=False).astype(int)
region_df['sf_rank'] = region_df['sf_abs_r'].rank(ascending=False).astype(int)

top15 = region_df.nsmallest(15, 'ps_rank').sort_values('ps_rank')
y_pos = range(len(top15))
bar_colors = ['#FF8A65' if r else '#90CAF9' for r in top15['ad_relevant']]

ax2.barh(y_pos, top15['ps_abs_r'], height=0.4, color=bar_colors,
         alpha=0.8, label='PetSurfer |r|')
ax2.barh([y + 0.4 for y in y_pos], top15['sf_abs_r'], height=0.4,
         color=[c.replace('FF8A65', 'FFCCBC').replace('90CAF9', 'BBDEFB')
                for c in bar_colors],
         alpha=0.8, label='Stanford |r|', edgecolor='gray', linewidth=0.5)
ax2.set_yticks([y + 0.2 for y in y_pos])
ax2.set_yticklabels(top15['label'], fontsize=9)
ax2.set_xlabel('|r(tau SUVR, PACC slope)|', fontsize=10)
ax2.set_title('Top 15 Most Prognostic Regions\n(PetSurfer ranking, Stanford comparison)',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('results/micdm_novel_prognostic_ranking.png',
            bbox_inches='tight', dpi=150)
plt.show()

print('Prognostic ranking agreement: Spearman rho = {:.3f} (p = {:.1e})'.format(
    rho_prog, p_prog))

In [ ]:
# ── Statistical Summary ──
print('=' * 70)
print('SUMMARY: Pipeline Agreement vs Clinical Relevance')
print('=' * 70)

print('\n1. Cross-Pipeline Agreement (ICC):')
all_icc = region_df['icc']
ad_icc = region_df[region_df['ad_relevant']]['icc']
non_ad_icc = region_df[~region_df['ad_relevant']]['icc']
print('   All regions:      median ICC = {:.3f} [{:.3f}, {:.3f}]'.format(
    all_icc.median(), all_icc.min(), all_icc.max()))
print('   AD-relevant:      median ICC = {:.3f} [{:.3f}, {:.3f}]'.format(
    ad_icc.median(), ad_icc.min(), ad_icc.max()))
print('   Non-AD:           median ICC = {:.3f} [{:.3f}, {:.3f}]'.format(
    non_ad_icc.median(), non_ad_icc.min(), non_ad_icc.max()))

u_stat, u_p = stats.mannwhitneyu(ad_icc, non_ad_icc, alternative='two-sided')
print('   AD vs non-AD ICC: U={:.0f}, p={:.3f}'.format(u_stat, u_p))

print('\n2. Prognostic Value (|r| with PACC slope):')
print('   All regions:      median |r| = {:.3f}'.format(region_df['mean_abs_r'].median()))
print('   AD-relevant:      median |r| = {:.3f}'.format(
    region_df[region_df['ad_relevant']]['mean_abs_r'].median()))
print('   Non-AD:           median |r| = {:.3f}'.format(
    region_df[~region_df['ad_relevant']]['mean_abs_r'].median()))

print('\n3. Key Relationship: ICC vs Prognostic Value')
rho, p = spearmanr(region_df['mean_abs_r'], region_df['icc'])
print('   Spearman rho = {:.3f}, p = {:.3f}'.format(rho, p))

print('\n4. Prognostic Ranking Agreement Between Pipelines')
rho_rank, p_rank = spearmanr(region_df['ps_abs_r'], region_df['sf_abs_r'])
print('   Spearman rho = {:.3f}, p = {:.1e}'.format(rho_rank, p_rank))

print('\n5. MI-CDM Schema (Park et al. 2025):')
print('   Tables: image_occurrence ({:,}), image_feature ({:,})'.format(
    len(image_occurrence), len(image_feature)))
print('   Polymorphic event concept: {} (measurement.measurement_id)'.format(
    image_feature['image_feature_event_field_concept_id'].unique()[0]))
print('   Pipelines tracked: {}'.format(
    sorted(image_feature['alg_system'].unique())))

print('\n' + '=' * 70)

---
## Discussion

### What This Analysis Shows

This analysis maps the **joint landscape of pipeline agreement and clinical prognostic value** across 42 bilateral FreeSurfer brain regions in the A4/LEARN trial.

### Why MI-CDM Is Essential

| Operation | MI-CDM Approach | Without MI-CDM |
|-----------|----------------|----------------|
| Match same-scan measurements | Join on `image_study_UID` via `image_occurrence` | Date-matching heuristic (approximate, error-prone) |
| Filter by pipeline | `image_feature.alg_system` | String-parse `measurement_source_value` (fragile) |
| Verify pipeline coverage | Count `image_feature` by `alg_system` per `image_occurrence` | Not possible without provenance tracking |
| Link imaging to procedures | `image_occurrence.procedure_occurrence_id` | Manual date+concept matching |

### MI-CDM Schema Reference

Implementation follows Park et al. 2025 (J Imaging Inform Med, 37:899-908) with 2 extension tables:
- **`image_occurrence`**: One row per DICOM series, with synthetic study/series UIDs
- **`image_feature`**: Polymorphic bridge using `image_feature_event_field_concept_id` (1147330) to link to `measurement.measurement_id`, with `alg_system` URNs for pipeline provenance